# Week 11 Contest: Predict the Failures

**IST 707 | In-class machine learning contest | Groups of 3 to 4**

A factory has 40 anonymised sensor columns (`f01` to `f40`) from machines that log a reading every few minutes.
For every row, `label = 1` means the machine **failed within the next 24 hours**. Your job is to predict that on rows you
haven't seen yet.

You have no column dictionary, no domain expert and no second chance at the hidden labels. You do have everything
from Weeks 1 to 10.

## Rules

| | |
|---|---|
| **Metric** | **Macro F1** on hard 0/1 predictions (average of the F1 for class 0 and class 1) |
| **Submissions** | **3 per round** per group. Extras are ignored. Choose carefully. |
| **Leaderboard** | Shows a score on a *public* 30% of the test rows. Your **final score** uses the other, *private* 70%. |
| **Rows are ordered by `id`** | `id` is time. The test rows come *after* the training rows. |
| **Tools** | Anything you like. Be ready to explain every choice you make. |

## Schedule

| Time | |
|---|---|
| 0:00 | Rules, data walkthrough, form groups |
| 0:15 to 0:55 | **Round 1** |
| 0:55 | Leaderboard and break |
| 1:05 to 1:55 | **Round 2** (something changes) |
| 1:55 | Break |
| 2:05 to 2:25 | Final submission window |
| 2:25 to 2:45 | Private scores revealed. Each group explains what mattered. |

## How you are graded

- **50%** final private-set score
- **30%** your explanation: what did you find, why did it help, what didn't work? Be specific. "We used XGBoost" is not an answer.
- **20%** improvement between your Round 1 and final scores

A group with a lower score and a clear, correct explanation beats a group with a higher score that can't say why.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

GROUP_NAME = "CHANGE-ME"      # <- your group's name (letters, numbers and dashes)
DATA_DIR = "contest_data"     # <- folder containing the .csv.gz files (use "." on Colab after uploading)

## 2. Load the data

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv.gz")
test_r1 = pd.read_csv(f"{DATA_DIR}/test_round1.csv.gz")

print("train:", train.shape, "  test_round1:", test_r1.shape)
train.head()

## 3. First look

Take a few minutes as a group **before** fitting anything. Some things worth checking:

- How balanced is the label? What would a model that always predicts `0` score on accuracy? On macro F1?
- What do the columns look like? Ranges, missing values, odd values.
- Do any columns change over `id`?
- Are there groups of rows (machine types) hiding in the data? You have a whole week of clustering behind you.

In [ ]:
print("Failure rate:", train["label"].mean().round(4))
print("Accuracy of 'always 0':", (1 - train["label"].mean()).round(4))
print("Macro F1 of 'always 0':", round(f1_score(train["label"], np.zeros(len(train)), average="macro"), 4))

In [ ]:
features = [c for c in train.columns if c.startswith("f")]
train[features].describe().T[["count", "mean", "std", "min", "50%", "max"]].round(2)

In [ ]:
train[features].isna().mean().sort_values(ascending=False).head(10).round(3)

## 4. Baseline

This is a deliberately plain model: all the columns as they are, default settings. **Every group starts here. This
is not a good score. It is the bar to beat.**

> **Validation hint:** the test rows are *later in time* than the training rows. Think about whether a random
> train/test split is the right way to estimate your score. The baseline below splits by time.

In [ ]:
cut = int(len(train) * 0.8)
fit_part, val_part = train.iloc[:cut], train.iloc[cut:]

base = HistGradientBoostingClassifier(random_state=0)
base.fit(fit_part[features], fit_part["label"])

val_pred = base.predict(val_part[features])
print("Validation macro F1:", round(f1_score(val_part["label"], val_pred, average="macro"), 4))
print(classification_report(val_part["label"], val_pred, digits=3))

## 5. Your turn

Add cells below. Ideas to get you going (none of them is a hint about what is actually in the data):

- Look at every column against `id` and against `label`.
- Look at where the model is wrong. Which kind of row does it miss?
- Is 0.5 the right cut-off for turning a probability into a 0/1 prediction?
- Does clustering the rows (K-Means, GMM) give you anything useful?
- Which of the 40 columns matter? Which are noise?

Keep a log of **what you tried and what happened**. You'll need it for the explanation part.

## 6. Making a submission

In [ ]:
import os, re
from datetime import datetime

def make_submission(test_df, preds, group, round_name, out_dir="my_submissions"):
    """Validate predictions and write  <group>__<round>__<HHMMSS>.csv  ready to hand in."""
    preds = np.asarray(preds).astype(int)
    assert len(preds) == len(test_df), f"Expected {len(test_df)} predictions, got {len(preds)}"
    assert set(np.unique(preds)) <= {0, 1}, "Predictions must be hard 0/1 labels, not probabilities"
    group = re.sub(r"[^A-Za-z0-9]+", "-", group).strip("-")
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{group}__{round_name}__{datetime.now():%H%M%S}.csv"
    pd.DataFrame({"id": test_df["id"].values, "pred": preds}).to_csv(path, index=False)
    print(f"Wrote {path}  ({preds.mean():.1%} predicted positive)")
    return path

In [ ]:
# Example: submit the baseline. Replace `base` and the features with your own model and pipeline.
preds = base.predict(test_r1[features])
make_submission(test_r1, preds, GROUP_NAME, "round1")

Hand the file from `my_submissions/` to your instructor (upload it to the class submission folder).
You have **5 submissions in Round 1**. The baseline counts as one if you submit it.